In [1]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../Data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [2]:
def run_sql_file(path):
    with open(path,'r') as f:
        script=f.read()
    con.executescript(script)
    con.commit()
    print('ran',path)

In [3]:
run_sql_file(r"C:\Users\muhju\Downloads\airtel Telecom Analysis\SQL\06_customer_features.sql")

ran C:\Users\muhju\Downloads\airtel Telecom Analysis\SQL\06_customer_features.sql


In [4]:
q("""
SELECT
    connection_type,
    COUNT(*) AS total_subscribers,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS share_percentage
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,total_subscribers,share_percentage
0,Postpaid,4851,22.05
1,Prepaid,17149,77.95


In [5]:
q("""
SELECT
    plan_name,
    plan_type,
    price_inr,
    COUNT(*) AS total_subscribers
FROM customer_features
GROUP BY plan_name, plan_type, price_inr
ORDER BY total_subscribers DESC
LIMIT 5;
""")

,plan_name,plan_type,price_inr,total_subscribers
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


In [6]:
q("""
SELECT
    circle_name,
    region,
    COUNT(*) AS total_subscribers
FROM customer_features
GROUP BY circle_name, region
ORDER BY total_subscribers DESC
LIMIT 5;
""")

,circle_name,region,total_subscribers
0,Maharashtra,West,1428
1,Andhra Pradesh,South,1412
2,Mumbai,West,1395
3,Karnataka,South,1361
4,North East,East,1318


In [7]:
q("""
SELECT
    age_band,
    COUNT(*) AS subs_count,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS s_percentage
FROM customer_features
GROUP BY age_band
ORDER BY subs_count DESC;
""")

,age_band,subs_count,s_percentage
0,25-34,7467,33.94
1,18-24,5268,23.95
2,35-44,4824,21.93
3,45-54,2895,13.16
4,55+,1546,7.03


In [8]:
q("""
SELECT
    acquisition_channel,
    COUNT(*) AS subscriber_count,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS share_percentage
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subscriber_count DESC;
""")

,acquisition_channel,subscriber_count,share_percentage
0,Retailer,9287,42.21
1,Airtel App,6513,29.60
2,Website,3081,14.00
3,Call Center,1770,8.05
4,Doorstep,1349,6.13


In [9]:
q("""
SELECT
    device_type,
    os,
    COUNT(*) AS subscriber_count
FROM customer_features
GROUP BY device_type,os
ORDER BY device_type,os""")

,device_type,os,subscriber_count
0,Feature Phone,Other,3029
1,Smartphone,Android,15613
2,Smartphone,iOS,3358


In [10]:
#B1 · What is our average monthly revenue per subscriber,
# by the company's ARPU figure? — 

#STEPS
#1. Open customer_features.
#2. Average stated_arpu.
#3. Report it.

q(""" SELECT AVG(stated_arpu) FROM customer_features ;""")

,AVG(stated_arpu)
0,421.645818


In [11]:
#B2 · What is our average monthly revenue per subscriber, from the payments received?
#STEPS
#1. Open customer_features again.
#2. Average actual_arpu_6m, which Step 2 built from recharges.
#3. Also total it, to get the monthly revenue of the whole base.

q(""" SELECT AVG(actual_arpu_6m), SUM(actual_arpu_6m) FROM customer_features ;""")

,AVG(actual_arpu_6m),SUM(actual_arpu_6m)
0,344.464097,7578210.14


In [12]:
#B3 · By how much does the company's ARPU figure overstate the money received?
#STEPS
#1. Average both columns in one query.
#2. Average the gap.
#3. Express the difference as a percentage of the actual total, not of the stated one.

q(""" SELECT AVG(stated_arpu),
         AVG(actual_arpu_6m),
         AVG(stated_arpu) - AVG(actual_arpu_6m) AS gap,
         (AVG(stated_arpu) - AVG(actual_arpu_6m)) / AVG(actual_arpu_6m) * 100 AS percentage_overstatement FROM customer_features ;""")

,AVG(stated_arpu),AVG(actual_arpu_6m),gap,percentage_overstatement
0,421.645818,344.464097,77.181721,22.406318


In [13]:
#B4 · Is that gap the same for prepaid and for postpaid subscribers?
#STEPS
#1. Group by connection_type.
#2. Average both columns inside each group.
#3. Read the two rows against each other rather than down the column.

q(""" SELECT connection_type,
         AVG(stated_arpu) AS avg_stated_arpu,
         AVG(actual_arpu_6m) AS avg_actual_arpu_6m,
         AVG(stated_arpu) - AVG(actual_arpu_6m) AS gap
 FROM customer_features
 GROUP BY connection_type ;""")

,connection_type,avg_stated_arpu,avg_actual_arpu_6m,gap
0,Postpaid,959.647289,970.418163,-10.770874
1,Prepaid,269.459385,167.398194,102.061191


In [14]:
#B5 · Which five plans bring in the most actual monthly revenue?
#STEPS
#1. Group by plan_name.
#2. Sum actual_arpu_6m for total monthly revenue and average it for revenue per subscriber.
#3. Sort by total, take five.

q(""" SELECT plan_name,
         SUM(actual_arpu_6m) AS total_monthly_revenue,
         AVG(actual_arpu_6m) AS avg_revenue_per_subscriber
 FROM customer_features
 GROUP BY plan_name
 ORDER BY total_monthly_revenue DESC
 LIMIT 5 ;""")

,plan_name,total_monthly_revenue,avg_revenue_per_subscriber
0,Postpaid Infinity 1499,2171756.20,1517.649336
1,Postpaid Family 999,1249245.94,1008.269524
2,Postpaid 749,718958.94,759.998879
3,Prepaid Entertainment 399,671849.50,313.655229
4,Prepaid Data 349,520066.00,276.043524


In [15]:
# B6 · Which five circles bring in the most actual monthly revenue?
#STEPS
#1. Group by circle_name.
#2. Sum actual_arpu_6m.
#3. Sort and take five.

q(""" SELECT circle_name,
         SUM(actual_arpu_6m) AS total_monthly_revenue
 FROM customer_features
 GROUP BY circle_name
 ORDER BY total_monthly_revenue DESC
 LIMIT 5 ;""")

,circle_name,total_monthly_revenue
0,Mumbai,473288.16
1,Karnataka,468514.95
2,Andhra Pradesh,463369.80
3,North East,460998.29
4,Maharashtra,450032.30


In [16]:
#B7 · How much money did we receive from January to June 2025?
#STEPS
#1. This is a question about payments, not subscribers, so read recharges.
#2. Filter to the window.
#3. Count and sum.

q(""" SELECT COUNT(*) AS total_recharges,
         SUM(amount_inr) AS total_revenue
    FROM recharges
   WHERE recharge_date >= '2025-01-01' AND recharge_date <= '2025-06-30';""")

,total_recharges,total_revenue
0,83693,45469268


In [17]:
#B8 · Do subscribers who have been with us longer pay more each month?
#STEPS
#1. Group by tenure_years.
#2. Average actual_arpu_6m.
#3. Sort by year

q(""" SELECT COUNT(*) AS total_subscribers,
        tenure_years,
        SUM(actual_arpu_6m) AS total_revenue,
        AVG(actual_arpu_6m) AS avg_revenue_per_subscriber
    FROM customer_features
   GROUP BY tenure_years
   ORDER BY avg_revenue_per_subscriber DESC;""")

,total_subscribers,tenure_years,total_revenue,avg_revenue_per_subscriber
0,3374,3,1205245.47,357.215611
1,7055,2,2470142.21,350.126465
2,6961,1,2385649.07,342.716430
3,4610,0,1517173.39,329.104857


AREA C

In [18]:
q(""" SELECT * FROM recharges;""")

,recharge_id,customer_id,recharge_date,amount_inr,plan_id,channel,txn_type
0,1,1,2023-04-23,299,4,Retailer,Recharge
1,2,1,2023-05-21,299,4,USSD,Recharge
2,3,1,2023-06-17,299,4,Website,Recharge
3,4,1,2023-07-17,299,4,Airtel App,Recharge
4,5,1,2023-08-12,299,4,Website,Recharge
...,...,...,...,...,...,...,...
408084,408085,21999,2024-11-24,399,6,Retailer,Recharge
408085,408086,21999,2024-12-20,399,6,USSD,Recharge
408086,408087,21999,2025-01-18,399,6,Website,Recharge
408087,408088,21999,2025-02-18,399,6,Airtel App,Recharge


In [19]:
#C1 · How many payments of each type did we take, and from how many subscribers?
#S T E P S
#1. Read recharges, where one row is one payment.
#2. Group by txn_type.
#3. Count rows and distinct subscribers in each group.

q(""" SELECT txn_type,
         COUNT(*) AS total_payments,
         COUNT(DISTINCT customer_id) AS total_subscribers
    FROM recharges
   GROUP BY txn_type;""")

,txn_type,total_payments,total_subscribers
0,Bill Payment,108419,4851
1,Recharge,299670,17149


In [20]:
#C2 · What is the average value of a payment? — deliberate failure 2

#S T E P S
#1. Read recharges.
#2. Average amount_inr.

q(""" SELECT AVG(amount_inr) AS avg_payment_value FROM recharges;""")

,avg_payment_value
0,532.58659


In [21]:
#C3 · What is the average payment for a prepaid recharge, and for a postpaid bill?
#S T E P S
#1. Same table, same average.
#2. Add GROUP BY txn_type.
#3. Add the minimum and maximum so the two ranges can be compared.

q(""" SELECT txn_type,
         AVG(amount_inr) AS avg_payment_value,
         MIN(amount_inr) AS min_payment_value,
         MAX(amount_inr) AS max_payment_value
    FROM recharges
   GROUP BY txn_type;""")

,txn_type,avg_payment_value,min_payment_value,max_payment_value
0,Bill Payment,1082.418239,399,1873
1,Recharge,333.660446,99,1799


In [22]:
#C4 · How many times did a prepaid and a postpaid subscriber pay in six months?
#S T E P S
#1. Back to customer_features, because this is now a question about subscribers.
#2. Group by connection_type.
#3. Average recharge_count and recharge_total_6m.

q(""" SELECT connection_type,
         AVG(recharge_count) AS avg_recharge_count,
         AVG(recharge_total_6m) AS avg_recharge_total_6m
    FROM customer_features
   GROUP BY connection_type;""")

,connection_type,avg_recharge_count,avg_recharge_total_6m
0,Postpaid,5.391878,5822.508555
1,Prepaid,3.355123,1004.389702


In [23]:
#C5 · Which payment channels take the most payments?
#S T E P S
#1. Read recharges.
#2. Group by channel.
#3. Count and share.

q(""" SELECT channel,
         COUNT(*) AS total_payments,
         COUNT(*) * 100.0 / (SELECT COUNT(*) FROM recharges) AS percentage
    FROM recharges
   GROUP BY channel
   ORDER BY total_payments DESC;""")

,channel,total_payments,percentage
0,Website,111163,27.239891
1,Airtel App,111083,27.220288
2,USSD,75257,18.441320
3,Retailer,74406,18.232787
4,AutoPay,36180,8.865713


In [24]:
#C6 · Do prepaid and postpaid subscribers pay through the same channels?
#STEPS
#1. Group by txn_type and channel.
#2. Use a window function to express each count as a share of its own txn_type, so the two groups can be 
#compared fairly.

q(""" SELECT txn_type,
         channel,
         COUNT(*) AS payment_count,
         COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY txn_type) AS percentage
    FROM recharges
   GROUP BY txn_type, channel
   ORDER BY txn_type, payment_count DESC;""")

,txn_type,channel,payment_count,percentage
0,Bill Payment,Airtel App,36186,33.376069
1,Bill Payment,AutoPay,36180,33.370535
2,Bill Payment,Website,36053,33.253397
3,Recharge,USSD,75257,25.113291
4,Recharge,Website,75110,25.064237
5,Recharge,Airtel App,74897,24.993159
6,Recharge,Retailer,74406,24.829312


In [25]:
q(""" SELECT * FROM customer_features;""")

,customer_id,activation_date,tenure_days,tenure_years,observed_days,circle_name,region,plan_name,plan_type,price_inr,...,actual_arpu_6m,stated_arpu,arpu_gap,complaint_count,unresolved_count,resolved_count,avg_resolution_days,r_score,f_score,m_score
0,9,2022-09-04,1030,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,290.83,374,83.17,3,0,3,9.33,2,3,3
1,31,2024-02-10,506,1,180,Delhi,North,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,3,2,3
2,151,2022-07-22,1074,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,349.00,374,25.00,4,1,3,5.33,2,3,3
3,186,2022-01-04,1273,3,180,Delhi,North,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,4,2,3
4,207,2024-12-14,198,0,91,Delhi,North,Prepaid Value 209,Prepaid,209,...,69.67,224,154.33,3,0,3,5.33,1,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,21805,2022-09-03,1031,2,180,North East,East,Prepaid Max 479,Prepaid,479,...,159.67,257,97.33,2,0,2,10.00,2,2,2
21996,21831,2022-12-18,925,2,180,North East,East,Prepaid Quarterly 719,Prepaid,719,...,239.67,257,17.33,0,0,0,NaN,2,2,3
21997,21906,2024-05-16,410,1,180,North East,East,Prepaid Entertainment 399,Prepaid,399,...,332.50,428,95.50,0,0,0,NaN,2,3,3
21998,21917,2024-02-25,491,1,180,North East,East,Prepaid Data 349,Prepaid,349,...,290.83,374,83.17,1,0,1,1.00,4,3,3


In [26]:
#C7 · Does the number of payments a subscriber made go with whether they left?
#STEPS
#1. Group customer_features by recharge_count.
#2. Count subscribers and average churn in each group.
#3. Restrict to counts of eight or fewer, where the groups are large enough to read.

q(""" SELECT recharge_count,
         COUNT(*) AS total_subscribers,
         AVG(is_churned) AS avg_churn
    FROM customer_features
   WHERE recharge_count <= 8
   GROUP BY recharge_count
   ORDER BY recharge_count;""")

,recharge_count,total_subscribers,avg_churn
0,0,3711,0.443007
1,1,2762,0.400434
2,2,2339,0.283454
3,3,1026,0.540936
4,4,714,0.760504
5,5,3407,0.137658
6,6,6732,0.015152
7,7,507,0.055227
8,8,32,0.906250


In [27]:
q(""" SELECT * FROM v_usage_month;""")

,customer_id,month,usage_days,total_mb,total_voice,roaming_days
0,1,2025-01,2,1222.5,60,0
1,1,2025-02,4,4496.3,114,0
2,1,2025-03,5,2871.0,50,0
3,1,2025-04,6,3815.2,109,0
4,2,2025-01,7,1870.5,225,1
...,...,...,...,...,...,...
117827,22000,2025-02,20,29489.8,1435,2
117828,22000,2025-03,20,33024.3,1489,0
117829,22000,2025-04,17,23536.9,1170,1
117830,22000,2025-05,23,39269.1,1521,1


In [28]:
#D1 · How much data did the network carry each month, and for how many subscribers?
#STEPS
#1. Read v_usage_month, where one row is one subscriber in one month.
#2. Group by month.
#3. Sum the data and convert megabytes to terabytes.
#4. Also count the distinct subscribers who appear, because that number is going to matter.

q(""" SELECT month,
         ROUND(SUM(total_mb) / 1024 / 1024, 2) AS total_data_tb,
         COUNT(DISTINCT customer_id) AS total_subscribers
    FROM v_usage_month
   GROUP BY month""")

,month,total_data_tb,total_subscribers
0,2025-01,337.97,21302
1,2025-02,307.32,21162
2,2025-03,332.51,20366
3,2025-04,310.27,19320
4,2025-05,363.10,18356
5,2025-06,338.95,17326


In [29]:
#  D2 · What is a subscriber's average daily data use?
# STEPS
# 1. Read usage_daily.
# 2. Average data_mb.
# 3. Report it.
q(""" SELECT ROUND(AVG(data_mb), 2) AS avg_daily_data FROM usage_daily""")

,avg_daily_data
0,1572.87


In [30]:
#D3 · What is the average data per subscriber, per active day and per observed day?
#STEPS
#1. Read customer_features, where Step 2 already computed both.
#2. Average each.
#3. Divide one by the other.

q(""" SELECT ROUND(AVG(mb_per_active_day), 2) AS avg_data_per_active_day,
         ROUND(AVG(mb_per_observed_day), 2) AS avg_data_per_observed_day,
         ROUND(AVG(mb_per_active_day) / AVG(mb_per_observed_day), 2) AS ratio
    FROM customer_features;""")

,avg_data_per_active_day,avg_data_per_observed_day,ratio
0,1404.44,524.05,2.68


In [31]:
#D4 · Does data use per day change with the day of the week?
#STEPS
#1. Read usage_daily.
#2. Use strftime('%w', usage_date) to pull the weekday out.
#3. Group by it.
#4. Count rows and average the data.

q(""" SELECT strftime('%w', usage_date) AS weekday,
         COUNT(*) AS total_days,
         ROUND(AVG(data_mb), 2) AS avg_data_mb
    FROM usage_daily
   GROUP BY strftime('%w', usage_date)""")

,weekday,total_days,avg_data_mb
0,0,190368,1578.06
1,1,189914,1572.94
2,2,183823,1569.55
3,3,191088,1564.48
4,4,190764,1580.00
5,5,190542,1569.51
6,6,190240,1575.45


In [32]:
q("""
SELECT
    CASE
        WHEN roaming_days = 0 THEN 'No roaming'
        WHEN roaming_days <= 5 THEN 'Low roaming'
        WHEN roaming_days <= 15 THEN 'Medium roaming'
        ELSE 'High roaming'
    END AS roaming_group,

    COUNT(*) AS total_subscribers,
    ROUND(AVG(is_Churned), 2) AS avg_churn

FROM customer_features
GROUP BY roaming_group
ORDER BY avg_churn;
""")

,roaming_group,total_subscribers,avg_churn
0,Medium roaming,765,0.04
1,Low roaming,15468,0.17
2,No roaming,5767,0.45


In [33]:
q("""
SELECT
    connection_type,
    AVG(roaming_days) AS avg_roaming_days,
    COUNT(CASE WHEN roaming_days > 0 THEN 1 END) AS ever_roamed,
    COUNT(*) AS total_subscribers
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,avg_roaming_days,ever_roamed,total_subscribers
0,Postpaid,1.836941,3630,4851
1,Prepaid,1.789317,12603,17149


In [34]:
q("""
SELECT
    is_roaming,
    COUNT(*) AS total_days,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS day_share_percent,
    COUNT(DISTINCT usage_id	) AS total_subscribers
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,total_days,day_share_percent,total_subscribers
0,0,1287143,97.02,1287143
1,1,39596,2.98,39596


In [35]:
q("""select round(avg(100*total_sms/nullif(usage_days,0)),2)as_sms_per_day,
sum(total_sms=0)as never_used,
count(*)as subs
from customer_features
where usage_days>0""")

,as_sms_per_day,never_used,subs
0,59.41,654,21996
